In [1]:
import os
import pandas as pd
import numpy as np
from datetime import date
from dateutil.relativedelta import relativedelta

In [2]:
def cpn_schedule(maturity, settle):
    """Function to get the coupon schedule"""
    if settle >= maturity:
        raise ValueError("settle must be before maturity")
    k = 0
    while maturity - relativedelta(months=6 * (k + 1)) > settle:
        k += 1
    n = k + 1                                         # coupons strictly after settle
    next_cpn = maturity - relativedelta(months=6 * k)
    prev_cpn = maturity - relativedelta(months=6 * (k + 1))
    return prev_cpn, next_cpn, n

In [3]:
def gilt_price(coupon, ytm, ex_div_bdays=7,\
    settle=date(2026, 10, 10), maturity=date(2055, 10, 10)):
    """Function to get clean, dirty, accrued, dv01 (per 100 nominal)"""
    prev_cpn, next_cpn, n = cpn_schedule(maturity, settle)
    t = (next_cpn-settle).days # days from settlement to next coupon
    s = (next_cpn-prev_cpn).days # days in current coupon period

    ex_div_date = (pd.Timestamp(next_cpn)-pd.offsets.BDay(ex_div_bdays)).date()
    is_ex = settle>=ex_div_date
    if is_ex:
        ai = -(coupon/2)*(t/s)
    else:
        ai = (coupon/2)*((s-t)/s)

    v = 1/(1+(ytm/2))
    cf = np.zeros(n, dtype=float)
    e = np.zeros(n, dtype=float)

    for k in range(n):
        if k==0:
            if is_ex:
                cf[k] = 0
            else:
                cf[k] = (coupon/2)
            if n == 1:
                cf[k] += 100
        elif k == n-1:
            cf[k] = (coupon/2)+100
        else:
            cf[k] = (coupon/2)
        e[k] = ((t/s)+k)
    
    dirty_P = np.sum(cf*v**e)
    clean_P = dirty_P-ai

    dv01 = 1e-4*0.5*np.sum(e*cf*v**(e+1))
    mac_dur = np.sum((e/2)*cf*v**e)/dirty_P
    mod_dur = mac_dur/(1+(ytm/2))

    return {"clean": clean_P, "dirty": dirty_P,\
            "ai": ai, "dv01": dv01,\
            "mac_dur": mac_dur, "mod_dur": mod_dur}

In [4]:
def bump_reprice(settle, maturity, coupon, ytm, ex_div_days):
    """Function to test whether (P(y-1bp)-P(y+1bp))/2==DV01"""
    p_up = gilt_price(coupon=coupon, ytm=ytm+1e-4, ex_div_bdays=ex_div_days,\
            settle=settle, maturity=maturity)["dirty"]
    p_down = gilt_price(coupon=coupon, ytm=ytm-1e-4, ex_div_bdays=ex_div_days,\
                settle=settle, maturity=maturity)["dirty"]
    
    return (p_down-p_up)/2

In [5]:
# TEST
coupon, ytm, ex_div_bdays, settle, maturity = 4.875, 0.05383, 7, date(2026, 11, 26), date(2055, 12, 7) 
results = gilt_price(coupon=coupon, ytm=ytm, ex_div_bdays=ex_div_bdays,\
                settle=settle, maturity=maturity)
results

{'clean': np.float64(92.58397657966573),
 'dirty': np.float64(92.4374601862231),
 'ai': -0.14651639344262293,
 'dv01': np.float64(0.13794847549824657),
 'mac_dur': np.float64(15.325102878302047),
 'mod_dur': np.float64(14.923438530260096)}

In [ ]:
coupon, ytm, ex_div_bdays, settle, maturity = 4.25, 0.05, 7, date(2026, 9, 30), date(2036, 7, 31) 
results = gilt_price(coupon=coupon, ytm=ytm, ex_div_bdays=ex_div_bdays,\
                settle=settle, maturity=maturity)

In [31]:
1e-4

0.0001